# Analyse des aides PAC 2024

## Qu'est-ce que la PAC ?
La **Politique Agricole Commune (PAC)** est le système de subventions européennes versées annuellement aux agriculteurs. En Hauts-de-France, elle représente **30 à 40% du revenu agricole**.

## Particularité de ce dataset
Contrairement aux autres datasets (RPG, SGDBE), les données PAC ne viennent pas d'un seul fichier brut.  
Elles sont issues de **plusieurs sources officielles** qu'on assemble :

| Aide | Montant | Source |
|---|---|---|
| DPB (Aide de Base au Revenu) | 118 €/ha | data.gouv.fr — Aides PAC versées 2022, onglet Hypothèses |
| Éco-régime niveau supérieur | 62 €/ha | Arrêté du 25/09/2024, publié JO le 01/10/2024 |
| Éco-régime niveau de base (pdt) | 45 €/ha | Même arrêté JO 01/10/2024 |
| VBC pois protéagineux | 104 €/ha | SMAG — PAC 2023-2027 guide complet |
| VBC pomme de terre industrie | 130 €/ha | Telepac — Notice aides découplées PAC 2024 |
| SIE | supprimé | Non subventionné depuis réforme PAC 2023 |

## Fichier utilisé
`agriculture-france-aides-pac-campagne-2022.xlsx` — téléchargé sur data.gouv.fr  
Ce fichier contient les montants réels versés par exploitation en 2022, dont le DPB unitaire dans l'onglet Hypothèses.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Imports                                     ║
# ╚══════════════════════════════════════════════════════════╝

!pip install openpyxl matplotlib pandas numpy -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#fafafa'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

CULTURES_MODELE = ['ble_tendre','colza','betterave','pomme_de_terre',
                   'lin_fibre','pois_proteine','orge','mais_grain']
NOMS = {
    'ble_tendre':'Blé tendre','colza':'Colza','betterave':'Betterave',
    'pomme_de_terre':'Pomme de terre','lin_fibre':'Lin fibre',
    'pois_proteine':'Pois protéagineux','orge':'Orge','mais_grain':'Maïs grain',
}
HDF_DEPTS = ['02','59','60','62','80']
NOMS_DEPT = {'02':'Aisne','59':'Nord','60':'Oise','62':'Pas-de-Calais','80':'Somme'}
print('✓ Imports OK')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Upload du fichier                           ║
# ╚══════════════════════════════════════════════════════════╝

from google.colab import files
print('Uploader : agriculture-france-aides-pac-campagne-2022.xlsx')
uploaded = files.upload()
XLSX_FILE = list(uploaded.keys())[0]
print(f'✓ Fichier reçu : {XLSX_FILE}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Exploration du fichier brut                 ║
# ║                                                          ║
# ║  Le fichier contient 3 onglets :                         ║
# ║  • Aides PAC  : montants versés par exploitation         ║
# ║  • Hypothèses : montant unitaire DPB (118€/ha)           ║
# ║  • Filtres    : critères d'éligibilité appliqués         ║
# ╚══════════════════════════════════════════════════════════╝

import openpyxl
wb = openpyxl.load_workbook(XLSX_FILE, read_only=True, data_only=True)

print('=== STRUCTURE DU FICHIER ===')
print(f'Onglets disponibles : {wb.sheetnames}')
print()

# ── Onglet Hypothèses ──
ws_hyp = wb['Hypothèses']
print('=== ONGLET HYPOTHÈSES ===')
print('Contient le montant unitaire DPB utilisé pour calculer la SAU')
print()
for row in ws_hyp.iter_rows(values_only=True):
    vals = [v for v in row if v is not None]
    if vals:
        print(f'  {vals}')
print()

# ── Onglet Filtres ──
ws_fil = wb['Filtres']
print('=== ONGLET FILTRES ===')
print('Critères appliqués pour sélectionner les bénéficiaires')
print()
for row in ws_fil.iter_rows(values_only=True):
    vals = [v for v in row if v is not None]
    if vals:
        print(f'  {vals}')
print()

# ── Onglet Aides PAC — aperçu ──
ws = wb['Aides PAC']
rows = list(ws.iter_rows(values_only=True))
print('=== ONGLET AIDES PAC ===')
print(f'Nombre de lignes : {len(rows):,}')
print(f'Header : {rows[0]}')
print()
print('5 premières lignes :')
for r in rows[1:6]:
    print(f'  {r}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — Explication des colonnes                    ║
# ║                                                          ║
# ║  Le fichier contient des informations par exploitation   ║
# ║  (anonymisées) avec le montant DPB versé.                ║
# ╚══════════════════════════════════════════════════════════╝

EXPLICATIONS_PAC = {
    'Raison sociale':    'Nom de l\'exploitation (anonymisé en partie)',
    'Commune':           'Commune de l\'exploitation',
    'Code postal':       'Code postal — permet d\'extraire le département (2 premiers chiffres)',
    'Département':       'Code département (formule Excel =LEFT(code_postal, 2))',
    'Rubrique':          'Type d\'aide PAC (DPB, Jeunes Agriculteurs, Agriculture Bio...)',
    'Montant':           'Montant versé en euros pour cette aide',
    'SAU sur base DPB':  'Surface Agricole Utile estimée = Montant / DPB_unitaire',
}

print('=== EXPLICATION DES COLONNES ===')
print()
for col, expl in EXPLICATIONS_PAC.items():
    print(f'  {col:<22} : {expl}')
print()
print('Point important — colonne Département :')
print('  Dans le fichier brut, la colonne Département contient une FORMULE Excel')
print('  (=LEFT(C2,2)) et non une valeur calculée.')
print('  → On extrait le département nous-mêmes depuis le code postal.')
print()
print('Point important — colonne SAU sur base DPB :')
print('  SAU = Montant_DPB / 118  (DPB unitaire 2022 = 118 €/ha)')
print('  C\'est une ESTIMATION de la surface de l\'exploitation.')
print('  Source : onglet Hypothèses du fichier → DPB 2022 = 118 €/ha')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — Chargement et nettoyage                     ║
# ╚══════════════════════════════════════════════════════════╝

data = []
for r in rows[1:]:
    if r[0] and r[4] and r[5]:
        data.append({
            'raison_sociale': r[0],
            'commune':        r[1],
            'code_postal':    str(r[2]) if r[2] else '00000',
            'rubrique':       r[4],
            'montant':        float(r[5]) if r[5] else 0,
        })

df = pd.DataFrame(data)
df['dept'] = df['code_postal'].str.zfill(5).str[:2]

print(f'Lignes chargées : {len(df):,}')
print()
print('Types d\'aides disponibles :')
for rub, n in df['rubrique'].value_counts().items():
    print(f'  {rub:<55} : {n:,} lignes')
print()

# Filtrer sur le DPB uniquement
dpb = df[df['rubrique'].str.contains('DPB', na=False)].copy()
dpb_hdf = dpb[dpb['dept'].isin(HDF_DEPTS)]

DPB_HA = 118  # Source : onglet Hypothèses du fichier
dpb['sau_ha'] = dpb['montant'] / DPB_HA

print(f'Bénéficiaires DPB France  : {len(dpb):,}')
print(f'Bénéficiaires DPB HdF     : {len(dpb_hdf):,}')
print(f'DPB unitaire 2022         : {DPB_HA} €/ha')
print(f'  (Source : onglet Hypothèses, cellule C2)')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Analyse DPB par département HdF             ║
# ╚══════════════════════════════════════════════════════════╝

print('=== DPB 2022 PAR DÉPARTEMENT HdF ===')
print(f'DPB unitaire : {DPB_HA} €/ha (source : onglet Hypothèses)')
print()
print(f'{"Dept":<6} {"Nom":<18} {"N exploit.":>10} '
      f'{"Moy €/exploit":>14} {"SAU moy (ha)":>13} {"Total M€":>10}')
print('-'*75)

for dept in sorted(HDF_DEPTS):
    sub = dpb_hdf[dpb_hdf.dept == dept]
    if len(sub) == 0: continue
    moy   = sub.montant.mean()
    sau   = moy / DPB_HA
    total = sub.montant.sum() / 1e6
    nom   = NOMS_DEPT[dept]
    print(f'  {dept}    {nom:<16} {len(sub):>10,}  '
          f'{moy:>13,.0f} €  {sau:>11.0f} ha  {total:>8.1f} M€')

print('-'*75)
print(f'  HdF   {"TOTAL":<16} {len(dpb_hdf):>10,}  '
      f'{dpb_hdf.montant.mean():>13,.0f} €  '
      f'{dpb_hdf.montant.mean()/DPB_HA:>11.0f} ha  '
      f'{dpb_hdf.montant.sum()/1e6:>8.1f} M€')
print()
print('Observation : l\'Aisne (02) et l\'Oise (60) ont les exploitations')
print('les plus grandes (~175 ha en moyenne) — grandes plaines céréalières.')
print('Le Nord (59) a les exploitations les plus petites (~125 ha) en raison')
print('de la présence de maraîchage et cultures légumières plus intensives.')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 7 — Construction du référentiel PAC 2024        ║
# ║                                                          ║
# ║  On construit la table PAC par culture en assemblant     ║
# ║  toutes les sources officielles.                         ║
# ╚══════════════════════════════════════════════════════════╝

PAC_AIDES = {
    # Source DPB : data.gouv.fr, Aides PAC 2022, onglet Hypothèses
    # Source Éco : Arrêté 25/09/2024, JO 01/10/2024
    # Source VBC : SMAG PAC 2023-2027 / Telepac notice 2024
    'ble_tendre':     {'dpb':118, 'eco':62,  'vbc':0  },
    'colza':          {'dpb':118, 'eco':62,  'vbc':0  },
    'betterave':      {'dpb':118, 'eco':62,  'vbc':0  },
    'orge':           {'dpb':118, 'eco':62,  'vbc':0  },
    'mais_grain':     {'dpb':118, 'eco':62,  'vbc':0  },
    'lin_fibre':      {'dpb':118, 'eco':62,  'vbc':0  },
    'pois_proteine':  {'dpb':118, 'eco':62,  'vbc':104},
    'pomme_de_terre': {'dpb':118, 'eco':45,  'vbc':130},
}

print('=== RÉFÉRENTIEL PAC 2024 PAR CULTURE ===')
print()
print(f'{"Culture":<22} {"DPB":>6} {"Éco-rég":>8} {"VBC":>6} {"TOTAL":>8}  Notes')
print('-'*72)
for c, v in PAC_AIDES.items():
    total = sum(v.values())
    note = '← aide couplée protéagineux' if v['vbc'] == 104 \
           else '← VBC industrie + éco réduit' if v['vbc'] == 130 else ''
    print(f'  {NOMS[c]:<20} {v["dpb"]:>6} {v["eco"]:>8} '
          f'{v["vbc"]:>6} {total:>8} €/ha  {note}')

print()
print('Points importants :')
print('  1. Le DPB est IDENTIQUE pour toutes les cultures (118 €/ha)')
print('     → pas discriminant entre cultures')
print('  2. L\'éco-régime est quasi identique (62€) sauf pdt (45€)')
print('     → peu discriminant')
print('  3. Le VBC est discriminant :')
print('     pois = 104€, pdt = 130€, toutes autres = 0€')
print('     → signal fort pour identifier pois et pdt')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 8 — Impact PAC sur le revenu net par culture    ║
# ║                                                          ║
# ║  On calcule le revenu net par culture en intégrant       ║
# ║  les aides PAC pour montrer leur impact réel.            ║
# ╚══════════════════════════════════════════════════════════╝

# Rendements HdF (Agreste SAA 2023-2024 × coeff régional)
RENDEMENTS_HDF = {
    'ble_tendre':7.54,'colza':3.39,'betterave':81.37,'pomme_de_terre':44.00,
    'lin_fibre':6.11,'pois_proteine':3.64,'orge':6.70,'mais_grain':9.44,
}
PRIX_TONNE = {
    'ble_tendre':220,'colza':480,'betterave':38,'pomme_de_terre':145,
    'lin_fibre':320,'pois_proteine':280,'orge':185,'mais_grain':195,
}
INTRANTS = {
    'ble_tendre':520,'colza':590,'betterave':760,'pomme_de_terre':960,
    'lin_fibre':380,'pois_proteine':290,'orge':420,'mais_grain':610,
}

print('=== IMPACT DE LA PAC SUR LE REVENU NET ===')
print(f'{"Culture":<22} {"CA brut":>9} {"PAC":>7} {"Charges":>9} '
      f'{"Revenu net":>11} {"% PAC/revenu":>14}')
print('-'*78)

resultats = {}
for c in CULTURES_MODELE:
    ca    = RENDEMENTS_HDF[c] * PRIX_TONNE[c]
    pac   = sum(PAC_AIDES[c].values())
    ch    = INTRANTS[c]
    rev   = ca + pac - ch
    pct   = pac / rev * 100 if rev > 0 else 0
    resultats[c] = {'ca':ca, 'pac':pac, 'charges':ch, 'revenu':rev, 'pct_pac':pct}
    print(f'  {NOMS[c]:<20} {ca:>9,.0f} {pac:>7} {ch:>9}  '
          f'{rev:>10,.0f} €  {pct:>12.1f}%')

print()
print('Exemple clé — Pois protéagineux SANS vs AVEC PAC :')
c = 'pois_proteine'
ca  = RENDEMENTS_HDF[c] * PRIX_TONNE[c]
ch  = INTRANTS[c]
pac = sum(PAC_AIDES[c].values())
print(f'  Sans PAC : {ca:.0f} - {ch} = {ca-ch:.0f} €/ha')
print(f'  Avec PAC : {ca:.0f} + {pac} - {ch} = {ca+pac-ch:.0f} €/ha')
print(f'  → La PAC représente {pac/(ca+pac-ch)*100:.0f}% du revenu net du pois')
print(f'  → Sans PAC, le pois serait économiquement peu attractif')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 9 — Visualisations                              ║
# ╚══════════════════════════════════════════════════════════╝

COULEURS = {
    'ble_tendre':'#f59e0b','mais_grain':'#f97316','orge':'#84cc16',
    'betterave':'#8b5cf6','pomme_de_terre':'#d97706','colza':'#eab308',
    'lin_fibre':'#3b82f6','pois_proteine':'#22c55e',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

noms_c = [NOMS[c] for c in CULTURES_MODELE]
cols   = [COULEURS[c] for c in CULTURES_MODELE]

# ── 1. Total PAC par culture ──
ax = axes[0]
totaux = [sum(PAC_AIDES[c].values()) for c in CULTURES_MODELE]
bars = ax.bar(noms_c, totaux, color=cols, edgecolor='white')
ax.set_ylabel('Aides PAC totales (€/ha)')
ax.set_title('Aides PAC 2024 par culture', fontsize=12, pad=10)
ax.tick_params(axis='x', rotation=35, labelsize=9)
for bar, t in zip(bars, totaux):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            f'{t}€', ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0, max(totaux)*1.2)

# ── 2. Décomposition DPB / Éco / VBC (stacked) ──
ax = axes[1]
dpbs = [PAC_AIDES[c]['dpb'] for c in CULTURES_MODELE]
ecos = [PAC_AIDES[c]['eco'] for c in CULTURES_MODELE]
vbcs = [PAC_AIDES[c]['vbc'] for c in CULTURES_MODELE]
x = range(len(CULTURES_MODELE))
ax.bar(x, dpbs, label='DPB (118€)', color='#93c5fd', edgecolor='white')
ax.bar(x, ecos, bottom=dpbs, label='Éco-régime', color='#86efac', edgecolor='white')
ax.bar(x, vbcs, bottom=[d+e for d,e in zip(dpbs,ecos)],
       label='VBC couplé', color='#fca5a5', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(noms_c, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('€/ha')
ax.set_title('Décomposition des aides PAC\nDPB + Éco-régime + VBC', fontsize=12, pad=10)
ax.legend(fontsize=9)

# ── 3. Revenu net avec et sans PAC ──
ax = axes[2]
rev_sans = [resultats[c]['ca'] - resultats[c]['charges'] for c in CULTURES_MODELE]
rev_avec = [resultats[c]['revenu'] for c in CULTURES_MODELE]
x = np.arange(len(CULTURES_MODELE))
w = 0.35
ax.bar(x-w/2, rev_sans, w, label='Sans PAC', color='#e5e7eb', edgecolor='white')
ax.bar(x+w/2, rev_avec, w, label='Avec PAC', color=cols, edgecolor='white', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(noms_c, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Revenu net (€/ha)')
ax.set_title('Revenu net avec et sans PAC\n(rendements HdF Agreste 2023-2024)', fontsize=12, pad=10)
ax.legend(fontsize=9)
ax.axhline(0, color='gray', linewidth=0.8)

plt.suptitle('Analyse des aides PAC 2024 — Impact sur les cultures HdF', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('pac_analyse_complete.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ pac_analyse_complete.png sauvegardé')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 10 — Intégration dans le modèle ML              ║
# ║                                                          ║
# ║  Explique comment la PAC est utilisée dans le modèle     ║
# ║  et pourquoi on a évité le data leakage.                 ║
# ╚══════════════════════════════════════════════════════════╝

print('=== INTÉGRATION PAC DANS LE MODÈLE ML ===')
print()
print('1. Dans le calcul du revenu_net_ha (target du régresseur) :')
print('   revenu_net_ha = (rendement × prix_tonne) + PAC_total - charges')
print('   → Les vrais montants PAC sont utilisés pour calculer le revenu réaliste')
print()
print('2. Comme features directes du classificateur (sans data leakage) :')
print()
print('   ⚠️  IMPORTANT — Data leakage évité :')
print('   Une version initiale utilisait pac_vbc_ha calculé DEPUIS culture_optimale')
print('   → corrélation 0.69 avec la cible → accuracy artificielle de 93%')
print('   → CORRIGÉ : la PAC est encodée sans référence à la culture cible')
print()
print('   Features PAC retenues (sans leakage) :')
features_pac = [
    ('pac_dpb_fixe',          118,  'Constante — DPB identique toutes cultures'),
    ('pac_eco_base',           45,  'Constante — éco-régime niveau de base'),
    ('pac_eco_superieur',      62,  'Constante — éco-régime niveau supérieur'),
    ('pac_vbc_pdt_possible', '0/1', 'Budget >= 900€/ha → peut cultiver pdt'),
    ('pac_vbc_pois_possible','0/1', 'Budget >= 250€/ha → peut cultiver pois'),
    ('pac_max_atteignable', 'calc', 'PAC max selon budget disponible'),
    ('pac_attractivite',    'calc', '(118+62) / budget_intrants'),
]
for f, v, d in features_pac:
    print(f'   {f:<28} valeur={str(v):<7}  {d}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 11 — Synthèse et téléchargements                ║
# ╚══════════════════════════════════════════════════════════╝

print('=' * 60)
print('  SYNTHÈSE — AIDES PAC 2024')
print('=' * 60)
print()
print('  Sources utilisées :')
print('    DPB 118€/ha    → data.gouv.fr, Aides PAC 2022, onglet Hypothèses')
print('    Éco 62€/ha     → Arrêté 25/09/2024, publié JO 01/10/2024')
print('    Éco 45€/ha     → Même arrêté (pdt — niveau de base)')
print('    VBC pois 104€  → SMAG, PAC 2023-2027 guide complet')
print('    VBC pdt 130€   → Telepac, notice aides découplées PAC 2024')
print()
print(f'  Bénéficiaires DPB France  : {len(dpb):,}')
print(f'  Bénéficiaires DPB HdF     : {len(dpb_hdf):,}')
print(f'  SAU moyenne HdF           : {dpb_hdf.montant.mean()/DPB_HA:.0f} ha/exploitation')
print(f'  Total DPB versé HdF       : {dpb_hdf.montant.sum()/1e6:.0f} M€')
print()
print('  Rôle dans le modèle ML :')
print('    → Calcul de revenu_net_ha (target régresseur)')
print('    → Features pac_* sans data leakage (classificateur)')

files.download('pac_analyse_complete.png')
print('\n✓ Téléchargement lancé')